In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/UGC-5823_2_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/UGC-5823_2_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     373   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        54   (2094, 964)   float32   


In [3]:
image_cut = image[38:138, 0:1890]
image_error_cut = image_error[38:138, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0188
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5, params6, params7):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) +
        sersic_1d(x_hr, params6) +
        sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/PSF/Real_seeing_UGC5823_2.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, params6, params7, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5, params6, params7)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, params6, params7, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    purple_area = single_integral(x, params6, a, b)
    pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area + purple_area + pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, params6, params7, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area, (purple_area*100)/sum_area, (pink_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 7'].iloc[3] -sigma, fit_HA['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5845402670184596
1.6196890925783567
7.607385737014641
8.829774182337516
3.580725077549781
3.257156078901806
0.6119669696120797
0.6118202310182674
0.8733748290117713
1.244799911385442
1.5610873452593568
1.6169814091628067
3.245987438507009
3.25805144512821
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.662352e+01  8.676128e-01  5.465420e-02  1.009840e-01   
Green      2.896486e+00  9.878353e+01  1.990371e-01  3.603152e-01   
Magenta    2.609029e-01  2.130622e-01  9.598728e+01  5.626035e+01   
Orange     0.000000e+00  0.000000e+00  1.407549e-01  2.543606e+01   
Chocolate  7.746807e-40  7.945481e-34  1.103971e-15  3.206132e-10   
Purple     2.190896e-01  1.357918e-01  3.618271e+00  1.784229e+01   
Deeppink   0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

                 Peak 5        Peak 6        Peak 7  
Blue       6.227521e-02  8.793341e-03  5.233499e-03  
Green      2.285974e-01  3.472820e-02  2.223422e-02  
Magenta    1.216807e+01  1.

In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 7'].iloc[3] -2*sigma, fit_HA['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.9241931458429855
2.9805269372458447
12.956835394199206
14.905862040497473
6.339671125727471
5.823324280249213
3.1985854134892153
3.146054793456556
1.768213559529291
2.360054151285193
7.091490692104401
7.27031598494059
5.973325500671456
5.993076610060033
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.627521e+01  1.034556e+00  6.210468e-02  9.756739e-02   
Green      3.202904e+00  9.855365e+01  2.265718e-01  3.486236e-01   
Magenta    2.838345e-01  2.518087e-01  9.516333e+01  6.494484e+01   
Orange     0.000000e+00  0.000000e+00  4.310603e-01  1.725046e+01   
Chocolate  1.256678e-39  1.585293e-33  6.280788e-15  1.783398e-07   
Purple     2.380544e-01  1.599818e-01  4.116936e+00  1.735851e+01   
Deeppink   0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

                 Peak 5     Peak 6        Peak 7  
Blue       6.180944e-02   0.009754  5.710351e-03  
Green      2.268283e-01   0.038470  2.423740e-02  
Magenta    1.222425e+01   1.328774  6.

In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'],
                      fit_HA['Component 7'], fit_HA['Component 7'].iloc[3] -3*sigma, fit_HA['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

4.377071699713828
4.443031912317585
16.75248360543551
18.999996109692905
8.797418015513824
8.195284545921194
4.9004419793246665
4.74226319902337
3.490284512658946
4.170220429993422
9.34018776696749
9.521258426340417
8.558821120144398
8.58226219451963
                 Peak 1        Peak 2        Peak 3     Peak 4        Peak 5  \
Blue       9.500674e+01  1.587881e+00  8.388534e-02   0.090127  5.297241e-02   
Green      4.353762e+00  9.787256e+01  3.090507e-01   0.322559  1.949561e-01   
Magenta    3.493315e-01  3.293173e-01  9.350295e+01  71.396503  1.045289e+01   
Orange     0.000000e+00  0.000000e+00  1.093767e+00  12.033024  7.596290e-04   
Chocolate  6.037684e-39  3.937147e-33  6.098268e-14   0.000148  1.579347e+00   
Purple     2.901673e-01  2.102435e-01  5.010344e+00  16.157639  8.771907e+01   
Deeppink   0.000000e+00  0.000000e+00  0.000000e+00   0.000000  3.478776e-14   

              Peak 6        Peak 7  
Blue        0.010456  6.753562e-03  
Green       0.041179  2.879994e-02

## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -sigma, fit_HA['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6129467501217933
0.6192955840538226
1.8232372541493096
1.9307304191709547
0.9433397232274439
0.9336297311481948
0.27271603455375637
0.27223453961480587
0.37726091717531535
0.43389335438994636
0.553903517156244
0.5623971415617045
1.2544021836813162
1.3015398806532512
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.621019e+01  3.447279e+00  2.279337e-01  2.483986e-01   0.155777   
Green      1.314399e+00  9.452745e+01  1.231850e-01  1.137912e-01   0.057506   
Magenta    2.367002e+00  1.893065e+00  9.525776e+01  8.609641e+01  35.463946   
Orange     1.061377e-77  1.029377e-40  8.370594e-03  3.627365e-01   0.002350   
Chocolate  1.810258e-23  1.074023e-21  1.522819e-14  3.922203e-12   0.002301   
Purple     1.084060e-01  1.322046e-01  4.382387e+00  1.317556e+01  64.179905   
Deeppink   2.234804e-09  3.756046e-08  3.592813e-04  3.107366e-03   0.138214   

                 Peak 6        Peak 7  
Blue       2.620937e-02  1.403972e-02  
Green     

In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -2*sigma, fit_HA['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)
df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.189274868715766
1.2030934516579217
3.2991166367591296
3.4581984707536226
1.7859083712327959
1.76622391561847
1.4637901374901883
1.4555668577990324
0.7650915111492484
0.8815774215012976
2.5969766788457114
2.632400436242602
2.3819067750807266
2.4553330834424356
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.605549e+01  3.858471e+00  2.421962e-01  2.337030e-01   0.154373   
Green      1.388417e+00  9.389662e+01  1.317340e-01  1.079770e-01   0.057140   
Magenta    2.443950e+00  2.098153e+00  9.494535e+01  8.687238e+01  35.261390   
Orange     3.669385e-75  7.235102e-39  1.382573e-02  2.934004e-01   0.004596   
Chocolate  1.959885e-23  1.281762e-21  2.441196e-14  2.359166e-11   0.002472   
Purple     1.121468e-01  1.467584e-01  4.666492e+00  1.248924e+01  64.369267   
Deeppink   2.377814e-09  4.309777e-08  4.036804e-04  3.293508e-03   0.150763   

                 Peak 6        Peak 7  
Blue       2.818036e-02  1.485038e-02  
Green      9.2678

In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -3*sigma, fit_HA['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


1.9736173535741557
1.9963739802760756
4.482427994932084
4.649623160629079
2.693434330874108
2.6658991456792482
2.1523942826457727
2.1357188042613284
1.4495651595380912
1.632936399961645
3.458973152055539
3.5057322721192756
3.6307656414858043
3.7156456544880174
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.547482e+01  5.549176e+00  3.008928e-01  2.249310e-01   0.137733   
Green      1.743299e+00  9.163242e+01  1.702862e-01  1.049004e-01   0.050843   
Magenta    2.657406e+00  2.635526e+00  9.429095e+01  8.730433e+01  31.568982   
Orange     9.054440e-70  7.411521e-37  2.288034e-02  2.430931e-01   0.007931   
Chocolate  2.760444e-23  1.661554e-21  4.721583e-14  2.032869e-10   0.002302   
Purple     1.244752e-01  1.828782e-01  5.214529e+00  1.211916e+01  68.002105   
Deeppink   3.098575e-09  5.355288e-08  4.629736e-04  3.588676e-03   0.230106   

                 Peak 6        Peak 7  
Blue       2.988600e-02  1.649035e-02  
Green      9.86083

## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -sigma, fit_HA['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5071285150950742
0.5207746712017743
2.0157542256165346
1.9843172322585425
0.9912693929493708
0.9241854868970317
0.2290566337968136
0.25296346716321627
0.4933549621826926
0.5041278943801304
0.4007426419535258
0.40856064104970086
0.8136886586713654
0.7545297008106564
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.639525e+01  4.578158e+00  1.943337e+00  3.469306e+00   
Green      1.511876e+00  9.401944e+01  6.385523e-02  6.975411e-02   
Magenta    8.833286e-01  7.367674e-01  8.973899e+01  6.541106e+01   
Orange     8.815602e-05  7.319413e-05  3.951739e-02  5.265327e+00   
Chocolate  8.706801e-13  2.603171e-10  1.976353e-03  7.970351e-02   
Purple     1.209454e+00  6.655639e-01  8.212324e+00  2.570485e+01   
Deeppink   7.420682e-57  1.083151e-43  2.110817e-18  2.832951e-14   

                 Peak 5     Peak 6     Peak 7  
Blue       2.296298e+00   1.071376   0.870620  
Green      2.036557e-02   0.005017   0.002654  
Magenta    1.460451e+01   3.773694 

In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -2*sigma, fit_HA['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.9637966329716396
0.9856711834855787
3.484478289865498
3.4352625612579013
1.8217226417737014
1.7158350132001956
1.2058825515717595
1.2881980685255747
0.9632829453950263
0.9800569017800485
1.8715410112103368
1.8948309760177087
1.4899981557552873
1.396556068035916
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.615268e+01  5.320122e+00  2.117491e+00  3.301588e+00   2.354279   
Green      1.640292e+00  9.305189e+01  7.047588e-02  6.769762e-02   0.021050   
Magenta    9.323935e-01  8.565043e-01  8.876405e+01  6.819363e+01  15.126257   
Orange     9.304397e-05  8.516229e-05  7.376396e-02  3.688796e+00   0.022128   
Chocolate  1.014764e-12  3.382436e-10  2.599630e-03  1.161217e-01  24.723136   
Purple     1.274537e+00  7.714014e-01  8.971616e+00  2.463217e+01  57.753149   
Deeppink   3.322437e-56  5.168663e-43  8.621853e-18  2.828690e-13   0.000002   

              Peak 6     Peak 7  
Blue        1.148693   0.951573  
Green       0.005445   0.00

In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] +3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -3*sigma, fit_HA['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5307990068182649
1.5563560691920508
4.641075401208112
4.583579327892702
2.70479918862649
2.5857822441275777
1.7931876715302513
1.86417143301753
1.6803244683968137
1.6985255335806924
2.5108518821953205
2.5274344679342
2.1829393852380496
2.0772396537948046
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.526125e+01  7.361454e+00  2.560105e+00  3.115183e+00   2.323015   
Green      2.219212e+00  9.055743e+01  9.358569e-02  6.525712e-02   0.020301   
Magenta    1.074335e+00  1.090300e+00  8.690135e+01  7.052010e+01  14.644200   
Orange     1.069748e-04  1.084051e-04  1.655541e-01  2.714459e+00   0.024151   
Chocolate  1.833618e-12  4.684822e-10  3.581984e-03  1.663264e-01  19.252693   
Purple     1.445093e+00  9.907054e-01  1.027582e+01  2.341868e+01  63.735597   
Deeppink   1.611738e-54  3.454254e-42  4.637515e-17  1.351186e-12   0.000044   

              Peak 6     Peak 7  
Blue        1.200588   1.119560  
Green       0.005767   0.003359  


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/SII_fit.csv", index_col = 0)

In [21]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5,Component 6,Component 7
I_e,0.0182,0.0547,0.0290,3.9481,0.0449,0.0226,0.6328
r_e,20.9444,5.4340,16.5770,0.0000,0.0636,33.7633,0.2955
n,1.7733,2.1194,1.5224,4.1999,2.6501,1.7847,0.3631
x_0,14.2773,28.8410,57.0877,61.5756,71.2236,77.4035,83.1078


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -sigma, fit_HA['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5446833989368077
0.5522166055806249
1.3514681998454336
1.3296154363519985
0.7001576002389921
0.6896955607852601
0.22025930465530652
0.22037271246172732
0.3888157392674003
0.4386673171998391
0.3902682495349613
0.39807309822721343
0.9051194736756893
0.7633195348267039
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.491921e+01  4.882064e+00  1.062385e+00  1.275883e+00   0.877959   
Green      1.974135e+00  9.208708e+01  1.629271e-01  1.392440e-01   0.054925   
Magenta    9.996294e-01  1.406702e+00  8.435438e+01  6.701636e+01  22.930154   
Orange     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   
Chocolate  2.236936e-25  1.106339e-22  1.614520e-13  1.125820e-10   0.244549   
Purple     2.107026e+00  1.624159e+00  1.442030e+01  3.156851e+01  75.892414   
Deeppink   0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   

                 Peak 6        Peak 7  
Blue       2.893282e-01  1.868364e-01  
Green     

C:\Users\ISAFA\AppData\Local\Temp\ipykernel_17680\3564233926.py:10: RuntimeWarning: divide by zero encountered in divide
  return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -2*sigma, fit_HA['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

C:\Users\ISAFA\AppData\Local\Temp\ipykernel_17680\3564233926.py:10: RuntimeWarning: divide by zero encountered in divide
  return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))


1.0283661720037702
1.0404346466195575
2.380423435941615
2.345962615475024
1.3312752143459132
1.3146490291935813
1.1376738536665005
1.1375581439180433
0.7846060023145697
0.8771942494314613
1.8207935723691775
1.8413711278039995
1.606466603444363
1.3846214776275385
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.456019e+01  5.581220e+00  1.120110e+00  1.239930e+00   0.871745   
Green      2.143167e+00  9.096755e+01  1.733804e-01  1.372405e-01   0.054851   
Magenta    1.062459e+00  1.604684e+00  8.349255e+01  6.784247e+01  22.924263   
Orange     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   
Chocolate  2.599877e-25  1.439223e-22  3.071875e-13  1.308345e-09   0.213603   
Purple     2.234188e+00  1.846546e+00  1.521396e+01  3.078036e+01  75.935538   
Deeppink   0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   

              Peak 6        Peak 7  
Blue        0.310995  2.108516e-01  
Green       0.012821  

In [24]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -3*sigma, fit_HA['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

C:\Users\ISAFA\AppData\Local\Temp\ipykernel_17680\3564233926.py:10: RuntimeWarning: divide by zero encountered in divide
  return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))


1.6113477370785563
1.6255302701883492
3.283162202926173
3.242357381951509
2.0899632427660473
2.0700942877673216
1.6427444713720392
1.6405008992853198
1.4550987130212643
1.582868969414227
2.4440371842905506
2.4480609404034497
2.2764151934752466
2.029104635211652
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.334948e+01  7.604227e+00  1.297576e+00  1.207339e+00   
Green      2.851573e+00  8.812707e+01  2.146442e-01  1.357037e-01   
Magenta    1.245235e+00  1.971011e+00  8.195534e+01  6.857623e+01   
Orange     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Chocolate  4.613362e-25  1.981920e-22  7.263903e-13  2.055970e-08   
Purple     2.553711e+00  2.297692e+00  1.653244e+01  3.008072e+01   
Deeppink   0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

                  Peak 5     Peak 6        Peak 7  
Blue        8.036496e-01   0.325419  2.548034e-01  
Green       4.983613e-02   0.013542  7.782898e-03  
Magenta     2.086899e+01   5.

## OIII

In [25]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/OIII_fit.csv", index_col = 0)

In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -sigma, fit_HA['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.8581919150207512
0.8600863677829722
1.5725408439397426
1.9457626203003777
1.0270082723497518
1.0612454145199706
0.25892125548388767
0.2589570310995374
0.4331996457162226
0.4325986187732188
0.5883996150290881
0.5843146556118068
0.8878973689917559
1.7096714568696947
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.783800e+01  6.819847e+00  1.216596e+00  1.854394e+00   
Green      1.607780e+00  9.216250e+01  2.473616e-01  2.915891e-01   
Magenta    3.963769e-01  7.029963e-01  9.118329e+01  7.375886e+01   
Orange     3.563865e-31  6.239629e-29  4.409833e-11  3.406746e-09   
Chocolate  1.365780e-24  2.489965e-23  8.472853e-18  3.477688e-15   
Purple     1.578470e-01  3.146558e-01  7.352751e+00  2.409516e+01   
Deeppink   5.930498e-23  3.012863e-20  5.095899e-13  2.919849e-11   

                 Peak 5        Peak 6        Peak 7  
Blue       1.394700e+00  3.512866e-01  3.577390e-01  
Green      1.465442e-01  2.749048e-02  2.317212e-02  
Magenta    1.58017

In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -2*sigma, fit_HA['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.6458266292072878
1.653936112318351
2.84047723742996
3.4444361228715357
1.9185167420949538
1.9710581225716006
1.4295057903780715
1.4343625032157334
0.8764174744775644
0.875174388563609
2.817872742145944
2.8052312658955447
1.7624726461423943
3.102650845985713
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.771162e+01  7.609705e+00  1.305347e+00  1.685367e+00   
Green      1.708586e+00  9.125830e+01  2.671594e-01  2.675537e-01   
Magenta    4.145774e-01  7.821357e-01  9.050405e+01  7.592664e+01   
Orange     3.930661e-31  7.690143e-29  1.221230e-10  2.181825e-09   
Chocolate  1.456619e-24  2.866256e-23  1.316757e-17  1.764232e-11   
Purple     1.652178e-01  3.498612e-01  7.923441e+00  2.212044e+01   
Deeppink   6.772605e-23  3.747274e-20  6.969378e-13  4.568073e-11   

                 Peak 5        Peak 6        Peak 7  
Blue       1.380978e+00  3.677022e-01  3.609034e-01  
Green      1.456294e-01  2.892251e-02  2.343229e-02  
Magenta    1.584040e+01  

In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit_HA['Component 7'].iloc[3] -3*sigma, fit_HA['Component 7'].iloc[3] + 3*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

2.659869238541728
2.6788468123772065
3.944264059361099
4.6408290139986175
2.813240570851343
2.870029045524632
2.1482357027259495
2.165892294416511
1.660347768176429
1.6576865172997277
3.770598377894798
3.794077533804317
2.867208679231159
4.455057863364566
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.720587e+01  1.035414e+01  1.615230e+00  1.576195e+00   
Green      2.136815e+00  8.826400e+01  3.474456e-01  2.528513e-01   
Magenta    4.688575e-01  9.553088e-01  8.892148e+01  7.725559e+01   
Orange     5.856826e-31  1.015540e-28  2.725624e-10  1.553163e-09   
Chocolate  1.860807e-24  3.500528e-23  2.547130e-17  1.660982e-10   
Purple     1.884611e-01  4.265486e-01  9.115841e+00  2.091537e+01   
Deeppink   1.155287e-22  4.985384e-20  1.050791e-12  7.101638e-11   

                 Peak 5        Peak 6        Peak 7  
Blue       1.248261e+00  3.857903e-01  3.805662e-01  
Green      1.301346e-01  3.051512e-02  2.446495e-02  
Magenta    1.412670e+01  2.56